In [6]:
import requests
import json
import os
from dotenv import load_dotenv
from datetime import datetime

# 1. Charger les variables d'environnement (.env)
load_dotenv()

class INPIExtractor:
    def __init__(self):
        self.username = os.getenv("INPI_USERNAME")
        self.password = os.getenv("INPI_PASSWORD")
        self.base_url = "https://registre-national-entreprises.inpi.fr/api"
        self.token = None
        
        # S'assurer que le dossier de stockage existe dès le début
        self.raw_dir = "data/raw_data"
        if not os.path.exists(self.raw_dir):
            os.makedirs(self.raw_dir)
            print(f"📂 Dossier créé : {self.raw_dir}")

    def login(self):
        """Récupère le token d'authentification SSO"""
        print("🔐 Authentification en cours...")
        url = f"{self.base_url}/sso/login"
        try:
            response = requests.post(url, json={"username": self.username, "password": self.password})
            if response.status_code == 200:
                self.token = response.json().get('token')
                print("✅ Connexion réussie.")
                return True
            else:
                print(f"❌ Erreur login ({response.status_code}) : {response.text}")
                return False
        except Exception as e:
            print(f"❌ Erreur de connexion au serveur : {e}")
            return False

    def fetch_companies(self, naf_code="62.01Z", page_size=20):
        """Extrait les données entreprises et force l'écriture du fichier"""
        if not self.token:
            if not self.login(): return

        # L'URL qui a fonctionné chez toi
        url = f"{self.base_url}/companies"
        
        headers = {
            "Authorization": f"Bearer {self.token}",
            "Accept": "application/json"
        }
        
        # Paramètres de recherche
        params = {
            "q": f"codeNaf:{naf_code}",
            "pageSize": page_size,
            "page": 1
        }

        print(f"🚀 Tentative d'extraction sur : {url} ...")
        
        try:
            response = requests.get(url, headers=headers, params=params)
            
            if response.status_code == 200:
                print("✅ Réponse reçue du serveur !")
                data = response.json()
                
                # --- GÉNÉRATION DU FICHIER ---
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                clean_naf = naf_code.replace('.', '')
                filename = f"{self.raw_dir}/inpi_raw_{clean_naf}_{timestamp}.json"
                
                with open(filename, 'w', encoding='utf-8') as f:
                    json.dump(data, f, ensure_ascii=False, indent=4)
                
                if os.path.exists(filename):
                    print(f"💾 SUCCÈS : Le fichier est ici -> {filename}")
                    
                    # Correction du bug 'list' object has no attribute 'get'
                    if isinstance(data, list):
                        count = len(data)
                    else:
                        count = len(data.get('content', data.get('results', [])))
                    
                    print(f"📊 Nombre d'entreprises récupérées : {count}")
                return data
                
        except Exception as e:
            print(f"❌ Erreur lors de l'extraction : {e}")
            return None

# --- EXÉCUTION DU SCRIPT ---
if __name__ == "__main__":
    extractor = INPIExtractor()
    # On lance l'extraction pour le secteur informatique
    extractor.fetch_companies(naf_code="62.01Z")

🔐 Authentification en cours...
✅ Connexion réussie.
🚀 Tentative d'extraction sur : https://registre-national-entreprises.inpi.fr/api/companies ...
✅ Réponse reçue du serveur !
💾 SUCCÈS : Le fichier est ici -> data/raw_data/inpi_raw_6201Z_20260321_155521.json
📊 Nombre d'entreprises récupérées : 20
